# 4000 senaryo: precoder selection → 1 W × 512 Z

Her bankada mevcut `select_precoder_rank1` fonksiyonu **bir W** seçer:

1. Bağımsız bir BR/RU pilot kanal çifti üret.
2. Sabit referans RIS deseniyle $F_{pilot}=G\,diag(\gamma_{ref})H$ oluştur.
3. `SVD(F_pilot)` ile `V` elde et ve **mevcut selector** fonksiyonunu çağır.
4. Seçilen W'yi sabitle; mevcut 512 Z'nin tamamında MC, Wick ve yeni analitik varyansı hesapla.

W seçimi q05 veya varyans hatasına göre yapılmaz. Fonksiyon tam Type-I codebook üzerinde
çalışır; seçilen W eski 32 adayın dışında da olabilir. Bu seçim tek pilot kanal ve referans
Z içindir; bütün Z'lerde veya bütün kanal gerçekleşmelerinde küresel optimum olduğu iddia edilmez.

**Karşılaştırma amacı:** hatanın önceki 32-W havuzundaki bazı W'lerde yoğunlaşmasını incelemek.
Bu nedenle 512 Z havuzu önceki `prepare_teacher_bank(k_w=32)` akışıyla hazırlanır ve korunur.
32 W için MC/varyans değerlendirmesi yapılmaz; yalnızca seçilen W değerlendirilir.
Pilot örneği, varyans MC akışından bağımsızdır.

| Sütun | Hesap |
|---|---|
| `varEmp` | Seçili W ve Z için Python MC güç varyansı |
| `sigma2Wick` | Mevcut Wick fonksiyonu |
| `sigma2Hop` | Yeni Gaussian-hop koşullu varyans formülü |

Varsayılan: mevcut `environments_4000.csv`, **4000 banka × 1 W × 512 Z**, banka başına
64.000 değerlendirme kanal çifti. MATLAB dosyası veya öğrenilmiş model gerekmez.


## 1. Drive ve GitHub kodu

Ayrı deney dalı ve ayrı Colab repo klasörü kullanılır. 32-W koşusunun koduna ve çıktılarına dokunulmaz.


In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = "https://github.com/OsmanMertYilmaz/Phy_Informed_MIMO_RIS_Env.git"
REF = "codex/selected-precoder-variance"
IN_COLAB = "google.colab" in sys.modules
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT = Path("/content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env")
    REPO_DIR = Path("/content/ris_selected_precoder_variance")
else:
    REPO_DIR = Path.cwd()
    if REPO_DIR.name == "notebooks":
        REPO_DIR = REPO_DIR.parent
    PROJECT = REPO_DIR / "data"

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", REF, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
if not (REPO_DIR / "src/ris_env/selected_precoder_variance.py").exists():
    raise RuntimeError(f"Deney kodu bulunamadı: {REPO_DIR}. {REF} dalını kullanın.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "matplotlib"], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))
print("Kod:", REPO_DIR)


## 2. Aynı 4000 senaryo, tek seçili W

`REFERENCE_Z_INDEX=4`, mevcut havuzdaki dört anchor sonrasındaki ilk global-random Z'dir.
Pilot SVD bu tek referans Z ile alınır. Sonrasında W **değiştirilmeden** bütün 512 Z değerlendirilir.
`PILOT_SEED_OFFSET` başka bir bağımsız pilot seçimiyle hassasiyet deneyi yapmak içindir;
ilk koşuda 0 bırakın. Ayar değiştirince yeni `RUN_TAG` kullanın.

`BASELINE_ROOT`, önceki 32-W koşusunun sonuç klasörüdür. Henüz tamamlanan bankalarla
karşılaştırma yapılabilir; bulunmazsa tek-W deneyi yine çalışır.


In [ ]:
import gc, json, time
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from ris_env.teacher_pipeline import prepare_teacher_bank
from ris_env.variance_comparison import compare_prepared_bank, summarize_comparisons
from ris_env.selected_precoder_variance import select_one_precoder, compare_with_full_pool

ENV_CSV = PROJECT / "environments" / "environments_4000.csv"
RUN_TAG = "mc_wick_new_selected_precoder_4000_v1"
OUTPUT = PROJECT / "analytic_variance_comparison" / RUN_TAG

BANK_IDS = None                 # None: CSV'deki 4000 bankanın tamamı
REFERENCE_Z_INDEX = 4
PILOT_SEED_OFFSET = 0
BASELINE_ROOT = PROJECT / "analytic_variance_comparison" / "mc_wick_new_analytic_4000_a100_v2"
Z_INDICES = None                # None: mevcut 512 Z
N_MC = 64_000
MC_CHUNK = 1024
MC_CHECKPOINT_SAMPLES = 8192
W_CHUNK = 1
Z_CHUNK = 128
ANALYTIC_Z_CHUNK = 16
PARITY = True                  # complex128 / float64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not ENV_CSV.is_file():
    raise FileNotFoundError(f"Mevcut 4000 senaryo dosyasını ENV_CSV ile gösterin: {ENV_CSV}")
environments = pd.read_csv(ENV_CSV)
if len(environments) != 4000 or environments.bankID.duplicated().any():
    raise ValueError("4000 benzersiz bankID içeren ortam CSV'si bekleniyor.")
if BANK_IDS is not None:
    missing = set(BANK_IDS) - set(environments.bankID)
    if missing:
        raise ValueError(f"Bilinmeyen bankID: {missing}")
    environments = environments[environments.bankID.isin(BANK_IDS)].copy()
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Cihaz:", DEVICE, "| Hassasiyet:", "float64" if PARITY else "float32")
print("Bankalar:", len(environments), "| MC/banka:", N_MC)
print("W: pilot SVD + mevcut precoder selector ile 1 codeword")
print("Pilot referans Z:", REFERENCE_Z_INDEX)
print("Z:", "512 (tümü)" if Z_INDICES is None else Z_INDICES)
print("Çıktı:", OUTPUT)


## 3. Precoder seçimi ve 1 W × 512 Z karşılaştırması

Pilot, Z havuzu, selector sonucu ve değerlendirme için kullanılan WIdx kaydedilir.
MC her 8192 örnekte, analitik sonuç seçili W tamamlandığında kaydedilir.
Aynı ayarlarla devam edilebilir. `wCandidate=0` bu deneyde yerel indekstir;
eski 32-W havuzundaki W=0 anlamına gelmez. Gerçek codeword `i11/i12/i2` ile tanımlanır.

`bank_summary.csv` bütün 512 Z üzerinden medyan/P90/P95/P99 hataları içerir.
`baseline_comparison.csv` mevcut 32-W çıktılarıyla banka/Z eşleşmesi sağlanan karşılaştırmadır.
Aynı codeword eski havuzdaysa eski MC akışının kendi içinde seçili-W / 32-W farkı da verilir.
Bu son karşılaştırma, yeni MC akışından kaynaklanabilecek değişimi ortadan kaldırır.


In [ ]:
summaries = []
plot_samples = []
paired_summaries = []
started = time.perf_counter()
for number, (_, row) in enumerate(environments.iterrows(), 1):
    bank_id = int(row.bankID)
    print(f"\n[{number}/{len(environments)}] bankID={bank_id}: ortam ve adaylar hazırlanıyor", flush=True)
    prepared = prepare_teacher_bank(row, k_w=32, device=DEVICE, parity=PARITY, z_chunk=Z_CHUNK)
    original_prepared = prepared
    prepared, selection, pilot_arrays = select_one_precoder(
        original_prepared, reference_z_index=REFERENCE_Z_INDEX,
        pilot_seed_offset=PILOT_SEED_OFFSET,
    )
    print("Seçili WIdx:", prepared.WIdx[0].tolist(),
          "| Eski 32-W indeksi:", selection["selectedOriginalWCandidate"], flush=True)
    folder = OUTPUT / f"bank_{bank_id:06d}"
    result = compare_prepared_bank(
        prepared, folder, n_mc=N_MC, mc_chunk=MC_CHUNK,
        w_chunk=W_CHUNK, z_chunk=Z_CHUNK, analytic_z_chunk=ANALYTIC_Z_CHUNK,
        checkpoint_samples=MC_CHECKPOINT_SAMPLES,
        w_indices=None, z_indices=Z_INDICES,
    )
    selection_tmp = folder / "precoder_selection.json.tmp"
    selection_tmp.write_text(json.dumps(selection, indent=2))
    selection_tmp.replace(folder / "precoder_selection.json")
    with (folder / "selection_pilot.npz.tmp").open("wb") as handle:
        np.savez_compressed(handle, **pilot_arrays)
    (folder / "selection_pilot.npz.tmp").replace(folder / "selection_pilot.npz")
    summary = {"bankID": bank_id, **selection, **summarize_comparisons(result)}
    valid = result.analyticValid & (result.varEmp > 0)
    for prefix in ["wick", "hop"]:
        errors = result.loc[valid, prefix + "AbsErrorPct"]
        for q in [90, 95, 99]:
            summary[f"{prefix}P{q}AbsErrorPct"] = float(errors.quantile(q / 100))
        summary[f"{prefix}Above20Pct"] = 100 * float((errors > 20).mean()) if len(errors) else float("nan")
    selected_gamma = prepared.gamma if Z_INDICES is None else prepared.gamma[torch.tensor(Z_INDICES, device=DEVICE)]
    paired_summaries.append({"bankID": bank_id, **compare_with_full_pool(
        result, BASELINE_ROOT / f"bank_{bank_id:06d}", selected_gamma,
        selected_row=prepared.row_physical,
    )})
    paired_tmp = OUTPUT / "baseline_comparison.csv.tmp"
    pd.DataFrame(paired_summaries).to_csv(paired_tmp, index=False)
    paired_tmp.replace(OUTPUT / "baseline_comparison.csv")
    for name in ["scenario_BR", "scenario_RU", "nRIS", "splitID", "split"]:
        if name in row.index:
            summary[name] = row[name]
    summaries.append(summary)
    plot_samples.append(result.sample(min(64, len(result)), random_state=20260911))
    bank_summary = pd.DataFrame(summaries)
    summary_tmp = OUTPUT / "bank_summary.csv.tmp"
    bank_summary.to_csv(summary_tmp, index=False)
    summary_tmp.replace(OUTPUT / "bank_summary.csv")
    plot_tmp = OUTPUT / "plot_sample.csv.tmp"
    pd.concat(plot_samples, ignore_index=True).to_csv(plot_tmp, index=False)
    plot_tmp.replace(OUTPUT / "plot_sample.csv")
    print(f"Wick medyan mutlak hata: %{summary['wickMedianAbsErrorPct']:.3f} | "
          f"Yeni analitik: %{summary['hopMedianAbsErrorPct']:.3f} | "
          f"Geçersiz: {summary['invalidRows']} | Geçen: {(time.perf_counter()-started)/60:.1f} dk")
    del result, prepared, original_prepared, pilot_arrays, selected_gamma
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("Tamamlandı:", OUTPUT)
display(bank_summary)


## 4. Karşılaştırma özeti

Aşağıdaki tablo **seçilen tek W ve 512 Z** için banka bazında hata dağılımını gösterir.
Aynı bankadaki W/Z satırları bağımsız örnekler sayılmaz.
`analyticValid=False` satırlar, negatif/sanal artık veya Wick tutarlılık kontrolü nedeniyle
ayrı sayılır; ham yeni analitik değerler kırpılmadan Parquet dosyasında saklanır.
Sıfır MC varyansında yüzde hata tanımsızdır ve özetten çıkarılan satır sayısına dahildir.


In [ ]:
bank_summary = pd.read_csv(OUTPUT / "bank_summary.csv")
metric_columns = ["wickMedianAbsErrorPct", "hopMedianAbsErrorPct", "hopP95AbsErrorPct", "hopAbove20Pct"]
display(bank_summary[metric_columns].describe(percentiles=[.5, .9, .95]).T)
print("Geçersiz / yüzde hata hesaplanamayan satırlar:", int(bank_summary.invalidRows.sum()))
valid_banks = bank_summary.validRows > 0
print("Yeni analitiğin banka medyan hatasını düşürdüğü banka oranı: %",
      round(100 * (bank_summary.loc[valid_banks, "hopMedianAbsErrorPct"] <
                   bank_summary.loc[valid_banks, "wickMedianAbsErrorPct"]).mean(), 2))
for grouping in [["scenario_BR", "scenario_RU"], ["nRIS"]]:
    if all(name in bank_summary for name in grouping):
        print("Banka medyanlarının medyanı:", grouping)
        display(bank_summary.groupby(grouping)[metric_columns].median())

paired = pd.read_csv(OUTPUT / "baseline_comparison.csv")
if "baselineComparable" in paired:
    paired_valid = paired.loc[paired.baselineComparable == True].copy()
    print("Önceki 32-W koşusuyla karşılaştırılabilen bankalar:", len(paired_valid))
    columns = [c for c in ["bankID", "fullPoolHopMedianAbsErrorPct", "selectedHopMedianAbsErrorPct",
        "fullPoolHopP95", "selectedHopP95", "selectorWPresentInOldPool",
        "sameOldStreamSelectedHopMedianAbsErrorPct", "sameOldStreamSelectedHopP95"] if c in paired_valid]
    display(paired_valid[columns])
    if len(paired_valid):
        print("Seçili-W medyan hatası daha düşük olan banka oranı: %",
              100 * (paired_valid.selectedHopMedianAbsErrorPct < paired_valid.fullPoolHopMedianAbsErrorPct).mean())
else:
    print("32-W referans tabloları bulunamadı. BASELINE_ROOT yolunu kontrol edin.")


In [ ]:
import matplotlib.pyplot as plt
plot_data = pd.read_csv(OUTPUT / "plot_sample.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, column, title in zip(axes, ["sigma2Wick", "sigma2Hop"], ["Wick", "Yeni analitik"]):
    valid = plot_data.analyticValid & (plot_data.varEmp > 0) & (plot_data[column] > 0)
    d = plot_data.loc[valid]
    if len(d):
        ax.scatter(d.varEmp, d[column], s=5, alpha=.2)
        lo = min(d.varEmp.min(), d[column].min()); hi = max(d.varEmp.max(), d[column].max())
        ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set(xscale="log", yscale="log", xlabel="MC güç varyansı", ylabel="Analitik güç varyansı", title=title)
    ax.grid(True, which="both", alpha=.2)
fig.tight_layout()
fig.savefig(OUTPUT / "mc_wick_new_analytic.png", dpi=180)
plt.show()


## 5. İstenilen bankayı ve MC yakınsamasını incele

Yakınsama tablosu aynı kanal akışının artan örnek sayılarındaki varyansıdır.
Bu noktalar bağımsız tekrarlar değildir; bu hücre yeni kanal üretmez.

`wCandidate=0` seçili W'dir. Önceki WIdx ile kıyaslamak için `precoder_selection.json` dosyasını kullanın.


In [ ]:
BANK_TO_INSPECT = int(bank_summary.bankID.iloc[0])
W_TO_INSPECT = 0  # Seçili tek W
Z_TO_INSPECT = 0 if Z_INDICES is None else int(Z_INDICES[0])
folder = OUTPUT / f"bank_{BANK_TO_INSPECT:06d}"
full = pd.read_parquet(folder / "comparison.parquet")
display(full[["bankID", "wCandidate", "zCandidate", "varEmp", "sigma2Wick", "sigma2Hop",
              "wickAbsErrorPct", "hopAbsErrorPct", "analyticValid"]].head(20))
manifest = json.loads((folder / "manifest.json").read_text())
ki = manifest["w_indices"].index(W_TO_INSPECT)
zi = manifest["z_indices"].index(Z_TO_INSPECT)
selected = full[(full.wCandidate == W_TO_INSPECT) & (full.zCandidate == Z_TO_INSPECT)].iloc[0]
curve = []
for path in sorted(folder.glob("mc_at_*.npz")):
    with np.load(path, allow_pickle=False) as data:
        curve.append({"N_MC": int(path.stem.split("_")[-1]), "varEmp": float(data["varEmp"][ki, zi]),
                      "sigma2Wick": selected.sigma2Wick, "sigma2Hop": selected.sigma2Hop})
convergence = pd.DataFrame(curve)
display(convergence)
if len(convergence):
    ax = convergence.plot(x="N_MC", y=["varEmp", "sigma2Wick", "sigma2Hop"], figsize=(8, 4), grid=True)
    ax.set_ylabel("Güç varyansı")
    plt.show()
